# Large-Scale News Article Classification Using NLP and Apache Spark MLlib — Logistic Regression

**Objective:** Automatically categorize news articles into **World, Sports, Business, and Sci/Tech** using NLP, TF-IDF, Apache Spark, and Spark MLlib.

**Important:** This is a classification/categorization project. The model's output for an unseen article is a predicted category; it is not a separate future-value forecasting problem.

### Pipeline
**Dataset → Spark DataFrame → Cleaning → Tokenization → Stop-word removal → TF-IDF → Classification → Evaluation → Performance analysis → New-article categorization**


## 1. Why this project?

News platforms can receive very large numbers of articles. Manually assigning every article to a topic is slow and difficult to scale.

This project demonstrates:

**Large dataset → distributed processing → NLP → feature extraction → machine learning → automatic categorization**

Apache Spark is used because Spark DataFrames support scalable data processing and Spark MLlib provides scalable machine-learning algorithms.

The notebook deliberately explains **what, why, and how** for each major step.


## 2. Dataset — AG News

We use the **AG News Topic Classification Dataset**. The standard benchmark contains **120,000 training samples and 7,600 test samples**, with four balanced classes: **World, Sports, Business, Sci/Tech**. Each record contains a class label, title, and description.

This is appropriate because it is a well-known text-classification benchmark and is large enough for an educational Spark project.

**Limitation:** 127,600 articles should not be called internet-scale Big Data. The project demonstrates a **Spark-based scalable architecture** that can be extended to much larger datasets.


In [ ]:
!pip -q install pyspark==3.5.6


In [ ]:
import os
import time
import urllib.request
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import RegexTokenizer, StopWordsRemover, HashingTF, IDF, StringIndexer
from pyspark.ml.classification import LogisticRegression, NaiveBayes
from pyspark.ml.evaluation import MulticlassClassificationEvaluator


In [ ]:
spark = (
    SparkSession.builder
    .appName("LargeScaleNewsClassification")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print("Spark version:", spark.version)
print("Spark master:", spark.sparkContext.master)
print("Default parallelism:", spark.sparkContext.defaultParallelism)


## 3. Download the dataset

We use the public AG News CSV files. The original train/test split is preserved rather than inventing a new benchmark split.

This is useful because the supplied benchmark already provides separate training and testing data.


In [ ]:
DATA_DIR = "/content/ag_news"
os.makedirs(DATA_DIR, exist_ok=True)

train_url = "https://raw.githubusercontent.com/mhjabreel/CharCnn_Keras/master/data/ag_news_csv/train.csv"
test_url  = "https://raw.githubusercontent.com/mhjabreel/CharCnn_Keras/master/data/ag_news_csv/test.csv"

train_path = os.path.join(DATA_DIR, "train.csv")
test_path = os.path.join(DATA_DIR, "test.csv")

if not os.path.exists(train_path):
    urllib.request.urlretrieve(train_url, train_path)
if not os.path.exists(test_path):
    urllib.request.urlretrieve(test_url, test_path)

print("Dataset files downloaded.")


In [ ]:
raw_train = (
    spark.read.option("header","false").option("inferSchema","true")
    .option("quote", '"').option("escape", '"').csv(train_path)
    .toDF("label_raw", "title", "description")
)

raw_test = (
    spark.read.option("header","false").option("inferSchema","true")
    .option("quote", '"').option("escape", '"').csv(test_path)
    .toDF("label_raw", "title", "description")
)

print("Training rows:", raw_train.count())
print("Test rows:", raw_test.count())
raw_train.printSchema()
raw_train.show(5, truncate=120)


## 4. Create the working Spark DataFrames

The original labels are integers 1–4. We map them to readable categories and combine the title with the description.

**Why combine them?** The title gives a short topical signal, while the description supplies additional context.


In [ ]:
label_map = {1:"World", 2:"Sports", 3:"Business", 4:"Sci/Tech"}
mapping_expr = F.create_map([F.lit(x) for pair in label_map.items() for x in pair])

train_df = (
    raw_train
    .withColumn("category", mapping_expr[F.col("label_raw")])
    .withColumn("text", F.concat_ws(" ", F.col("title"), F.col("description")))
    .select("category", "text")
)

test_df = (
    raw_test
    .withColumn("category", mapping_expr[F.col("label_raw")])
    .withColumn("text", F.concat_ws(" ", F.col("title"), F.col("description")))
    .select("category", "text")
)

train_df.show(5, truncate=120)


## 5. Data quality checks and cleaning

We check missing/empty text and duplicate articles.

**Why?**
- Missing text cannot provide useful features.
- Empty documents should not reach the classifier.
- Duplicate content can bias training/evaluation.


In [ ]:
print("Missing/empty train rows:",
      train_df.filter(F.col("text").isNull() | (F.trim(F.col("text")) == "")).count())
print("Missing/empty test rows:",
      test_df.filter(F.col("text").isNull() | (F.trim(F.col("text")) == "")).count())

print("Duplicate train rows:",
      train_df.count() - train_df.dropDuplicates(["text"]).count())
print("Duplicate test rows:",
      test_df.count() - test_df.dropDuplicates(["text"]).count())

train_df = (train_df
            .filter(F.col("text").isNotNull())
            .filter(F.length(F.trim(F.col("text"))) > 0)
            .dropDuplicates(["text"]))

test_df = (test_df
           .filter(F.col("text").isNotNull())
           .filter(F.length(F.trim(F.col("text"))) > 0)
           .dropDuplicates(["text"]))

print("Clean train rows:", train_df.count())
print("Clean test rows:", test_df.count())


## 6. Exploratory Data Analysis

The category distribution shows whether the dataset is balanced. AG News is designed with equal class sizes.


In [ ]:
category_pd = train_df.groupBy("category").count().orderBy(F.desc("count")).toPandas()
display(category_pd)

plt.figure(figsize=(8,5))
sns.barplot(data=category_pd, x="category", y="count")
plt.title("News Articles by Category")
plt.xlabel("Category")
plt.ylabel("Number of Articles")
plt.xticks(rotation=20)
plt.show()


In [ ]:
length_df = train_df.withColumn("word_count", F.size(F.split(F.col("text"), r"\s+")))
length_sample = length_df.select("category","word_count").sample(
    False, 0.05, seed=42
).toPandas()

plt.figure(figsize=(9,5))
sns.boxplot(data=length_sample, x="category", y="word_count")
plt.title("Article Word Count by Category (5% Sample)")
plt.xlabel("Category")
plt.ylabel("Word Count")
plt.xticks(rotation=20)
plt.show()


## 7. NLP preprocessing

**Raw text → lowercase/cleaning → tokenization → stop-word removal → TF-IDF**

- **Cleaning:** removes URLs, HTML, punctuation, and excess whitespace.
- **Tokenization:** converts a document into individual terms.
- **Stop-word removal:** removes very common words that usually carry little topic information.
- **TF-IDF:** converts the remaining text into numerical features.


In [ ]:
def clean_text(df):
    return (
        df
        .withColumn("text", F.lower(F.col("text")))
        .withColumn("text", F.regexp_replace(F.col("text"), r"http\S+|www\.\S+", " "))
        .withColumn("text", F.regexp_replace(F.col("text"), r"<[^>]+>", " "))
        .withColumn("text", F.regexp_replace(F.col("text"), r"[^a-z0-9\s]", " "))
        .withColumn("text", F.regexp_replace(F.col("text"), r"\s+", " "))
        .withColumn("text", F.trim(F.col("text")))
    )

train_clean = clean_text(train_df)
test_clean = clean_text(test_df)
train_clean.select("category","text").show(5, truncate=120)


In [ ]:
label_indexer = StringIndexer(inputCol="category", outputCol="label", handleInvalid="keep")
label_model = label_indexer.fit(train_clean)

train_labeled = label_model.transform(train_clean)
test_labeled = label_model.transform(test_clean)

print("Category -> numeric label:")
for i, label in enumerate(label_model.labels):
    print(i, "->", label)

train_labeled = train_labeled.repartition(8).cache()
test_labeled = test_labeled.repartition(8).cache()

print("Train partitions:", train_labeled.rdd.getNumPartitions())
print("Test partitions:", test_labeled.rdd.getNumPartitions())


## 8. TF-IDF feature extraction

Spark MLlib is used for the full feature pipeline.

**Why HashingTF?** It creates a fixed-size sparse representation without maintaining a huge explicit vocabulary, which is useful for scalable text processing.

**Why IDF?** It downweights terms that occur in many documents and gives more weight to informative terms.


In [ ]:
tokenizer = RegexTokenizer(
    inputCol="text", outputCol="tokens",
    pattern=r"\W+", minTokenLength=2
)

stopwords = StopWordsRemover(
    inputCol="tokens", outputCol="filtered_tokens"
)

hashing_tf = HashingTF(
    inputCol="filtered_tokens",
    outputCol="tf_features",
    numFeatures=1 << 18
)

idf = IDF(
    inputCol="tf_features",
    outputCol="features",
    minDocFreq=5
)


## 9. Spark MLlib Model — Logistic Regression

Logistic Regression is a linear classifier. It is a useful baseline for sparse text vectors because text classification often works well with linear decision boundaries.


In [ ]:
lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    maxIter=30,
    regParam=0.05,
    elasticNetParam=0.0
)

lr_pipeline = Pipeline(stages=[tokenizer, stopwords, hashing_tf, idf, lr])

start_lr = time.perf_counter()
lr_model = lr_pipeline.fit(train_labeled)
lr_train_time = time.perf_counter() - start_lr

print(f"Logistic Regression training time: {lr_train_time:.2f} seconds")


In [ ]:
evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction"
)

start = time.perf_counter()
lr_predictions = lr_model.transform(test_labeled).cache()
lr_predictions.count()
lr_prediction_time = time.perf_counter() - start

lr_accuracy = evaluator.evaluate(lr_predictions, {evaluator.metricName:"accuracy"})
lr_precision = evaluator.evaluate(lr_predictions, {evaluator.metricName:"weightedPrecision"})
lr_recall = evaluator.evaluate(lr_predictions, {evaluator.metricName:"weightedRecall"})
lr_f1 = evaluator.evaluate(lr_predictions, {evaluator.metricName:"f1"})

print(f"Accuracy:  {lr_accuracy:.4f}")
print(f"Precision: {lr_precision:.4f}")
print(f"Recall:    {lr_recall:.4f}")
print(f"F1 Score:  {lr_f1:.4f}")
print(f"Prediction time: {lr_prediction_time:.2f} seconds")


## 10. Confusion Matrix

A confusion matrix compares the **actual category** with the **predicted category**. The diagonal represents correct classifications; off-diagonal cells show where the model confused one topic with another.


In [ ]:
def show_confusion_matrix(predictions, title):
    cm = (predictions.groupBy("label","prediction").count()
          .orderBy("label","prediction").toPandas())

    n = len(label_model.labels)
    matrix = pd.DataFrame(0, index=range(n), columns=range(n))
    for _, row in cm.iterrows():
        matrix.loc[int(row["label"]), int(row["prediction"])] = int(row["count"])

    matrix.index = label_model.labels
    matrix.columns = label_model.labels

    plt.figure(figsize=(8,6))
    sns.heatmap(matrix, annot=True, fmt="d", cmap="Blues")
    plt.title(title)
    plt.xlabel("Predicted Category")
    plt.ylabel("Actual Category")
    plt.show()
    return matrix

lr_cm = show_confusion_matrix(lr_predictions, "Logistic Regression Confusion Matrix")


## 11. Spark / Big Data Demonstration

This section records Spark configuration, partitions, and a measured partition experiment.

**Important academic point:** Colab normally runs Spark locally. This demonstrates Spark's distributed processing model and partitioning, but it is not evidence of a multi-node production cluster. Do not fabricate a distributed speedup.


In [ ]:
print("Spark master:", spark.sparkContext.master)
print("Default parallelism:", spark.sparkContext.defaultParallelism)
print("Training partitions:", train_labeled.rdd.getNumPartitions())
print("Test partitions:", test_labeled.rdd.getNumPartitions())


In [ ]:
def benchmark_count(partitions):
    df = train_labeled.repartition(partitions)
    start = time.perf_counter()
    row_count = df.select("category","text").count()
    elapsed = time.perf_counter() - start
    return partitions, row_count, elapsed

benchmark_df = pd.DataFrame(
    [benchmark_count(2), benchmark_count(8)],
    columns=["Partitions","Rows","Time (s)"]
)

display(benchmark_df)


## 12. Performance Analysis

Training and prediction times below are measured on the current Colab runtime and therefore depend on available CPU, memory, Spark version, and runtime conditions.

Discuss:
- preprocessing cost
- TF-IDF cost
- model training time
- prediction time
- partitioning
- memory considerations
- how the same Spark pipeline could scale to larger datasets


In [ ]:
performance_summary = pd.DataFrame([
    {
        "Model": "Logistic Regression",
        "Training Time (s)": lr_train_time,
        "Prediction Time (s)": lr_prediction_time,
        "Accuracy": lr_accuracy,
        "Precision": lr_precision,
        "Recall": lr_recall,
        "F1 Score": lr_f1
    }
])

display(performance_summary.round(4))

## 13. Classify a New Unseen Article

This is the practical categorization use case. The trained Spark pipeline processes a new article and returns one of the four learned categories.


In [ ]:
def classify_news(article_text, model=lr_model):
    new_df = spark.createDataFrame([(article_text,)], ["text"])
    new_df = clean_text(new_df)
    row = model.transform(new_df).select("prediction","probability").first()

    predicted_index = int(row["prediction"])
    predicted_category = label_model.labels[predicted_index]
    return predicted_category, row["probability"]

example_article = (
    "NVIDIA introduced a new generation of processors designed to accelerate "
    "artificial intelligence workloads in data centers."
)

category, probabilities = classify_news(example_article)

print("Predicted Category:", category)
print("Class probabilities:", probabilities)


In [ ]:
# Replace this text with your own news article/headline.
user_article = (
    "Apple reported strong quarterly results as demand for its latest "
    "devices increased across several major markets."
)

category, probabilities = classify_news(user_article)

print("Article:", user_article)
print("\nPredicted Category:", category)
print("Probabilities:", probabilities)


## 14. Data-Driven Interpretation

The notebook must not automatically label one model "best." Use the actual measured F1/accuracy, confusion matrices, and training time to explain the trade-off.


# 15. Conclusion

The project automatically categorizes news articles using NLP and Spark MLlib.

**NLP:** text is cleaned, tokenized, filtered, and converted into TF-IDF vectors.

**Big Data:** Spark DataFrames and Spark MLlib provide a scalable processing/ML architecture.

**Models:** Logistic Regression is evaluated using accuracy, precision, recall, F1-score, and a confusion matrix.

**Limitation:** The AG News benchmark is substantial for an educational Colab project but is not internet-scale. The project should claim a scalable architecture rather than fake a production-scale cluster benchmark.

### Future scope
- Larger news corpora
- More categories
- Kafka + Spark Structured Streaming
- Multilingual classification
- Transformer/embedding features
- API/web deployment


# 16. Viva Preparation

1. **Why this project?** Automatic topic categorization reduces manual work for large news collections.
2. **Why Spark?** Distributed DataFrame processing and scalable MLlib algorithms.
3. **What is MLlib?** Spark's machine-learning library.
4. **What is NLP?** Processing human language for computational analysis.
5. **Why preprocessing?** Raw text must be cleaned and converted into numerical features.
6. **What is tokenization?** Splitting text into tokens.
7. **What are stop words?** Very common words with little topical information.
8. **What is TF-IDF?** A numerical representation that emphasizes informative terms.
9. **Why not raw text?** ML algorithms require numerical feature vectors.
10. **Why Logistic Regression?** A strong and simple linear classifier for sparse TF-IDF text features.
11. **What is classification?** Assigning an input article to a predefined category.
12. **Classification vs regression?** Categories versus continuous numeric outputs.
13. **Training data?** Data used to learn model parameters.
14. **Testing data?** Unseen data used to evaluate the trained model.
15. **Overfitting?** Learning training-specific patterns that do not generalize.
16. **Accuracy?** Fraction of correct predictions.
17. **Precision/Recall/F1?** Measures of correctness, coverage, and their harmonic combination.
18. **Confusion matrix?** Actual-vs-predicted class counts.
19. **How is it scalable?** Spark DataFrames and MLlib can execute across Spark workers.
20. **Distributed processing?** Splitting computation/data across resources.
21. **Why DataFrames?** Structured, optimized distributed processing.
22. **What are partitions?** Chunks of distributed data processed in parallel.
23. **Limitations?** Colab is not a production multi-node cluster; AG News is not internet-scale.
24. **How to make it real-time?** Kafka + Spark Structured Streaming can feed incoming articles to the trained classifier.


# 17. Suggested Report Screenshots

Capture:
1. Spark session/version
2. Dataset schema and record counts
3. Sample articles
4. Category distribution
5. Text preprocessing/TF-IDF pipeline
6. Logistic Regression metrics
8. Confusion matrix
9. Partition/performance analysis
10. New article → predicted category
11. Gradio Web UI
12. Final measured results

### Report structure
**Introduction → Problem Statement → Objectives → Dataset → Technology Stack → Architecture → Preprocessing → TF-IDF → Spark MLlib Models → Evaluation → Performance Analysis → Results → Conclusion → Future Scope**


# 18. Interactive Web UI — Gradio

This section provides a browser-based interface for testing the trained Spark MLlib Logistic Regression model. It uses the already-trained `lr_model` and `classify_news()` function, so **the Spark training cells do not need to be rerun when you only want to test the UI**.

Google Colab is supported by Gradio. With `share=True`, Gradio creates a browser-accessible share URL for the running app. citeturn0search0turn0search2

In [ ]:
%pip -q install -U gradio


In [ ]:
import gradio as gr

def predict_news(article):
    if article is None or not article.strip():
        return "Please enter a news article.", {}

    try:
        category, probability_vector = classify_news(article, lr_model)

        probabilities = {
            label_model.labels[i]: float(probability_vector[i])
            for i in range(len(label_model.labels))
        }

        return category, probabilities

    except Exception as e:
        return f"Error: {str(e)}", {}

with gr.Blocks(title="News Article Classifier — Spark MLlib") as demo:
    gr.Markdown("""
    # 📰 Large-Scale News Article Classifier
    ### Apache Spark MLlib + NLP + TF-IDF

    Enter a news article and the trained **Spark MLlib Logistic Regression** model
    will classify it into one of four AG News categories.
    """)

    article_input = gr.Textbox(
        label="News Article",
        placeholder="Paste a news article title and/or description here...",
        lines=12
    )

    classify_button = gr.Button("🔍 Classify Article", variant="primary")

    category_output = gr.Textbox(
        label="Predicted Category",
        interactive=False
    )

    probability_output = gr.Label(
        label="Class Probabilities",
        num_top_classes=4
    )

    classify_button.click(
        fn=predict_news,
        inputs=article_input,
        outputs=[category_output, probability_output]
    )

    gr.Examples(
        examples=[
            ["The national football team won the championship after defeating its rivals in a dramatic final."],
            ["The technology company announced a new artificial intelligence processor designed for data centers."],
            ["Stock markets rose sharply after investors reacted positively to the latest economic data."],
            ["The government announced a new international agreement following negotiations with world leaders."]
        ],
        inputs=article_input
    )

    gr.Markdown("**Categories:** World • Sports • Business • Sci/Tech")

demo.launch(share=True, debug=True)


## 19. Final Notes for Submission

- The main classifier is **Logistic Regression from Spark MLlib**.
- The NLP pipeline uses Spark transformations for cleaning, tokenization, stop-word removal, HashingTF, and IDF.
- Evaluation uses accuracy, weighted precision, weighted recall, F1-score, and a confusion matrix.
- The Gradio UI is only the presentation/testing layer; prediction still runs through the trained Spark MLlib model.
- Do not invent performance numbers. Use the values generated by the executed Colab runtime.
- The Colab runtime is a single-machine educational environment; describe the architecture as Spark-scalable rather than claiming a multi-node cluster benchmark.

# 20. References

- AG News dataset documentation: https://huggingface.co/datasets/sh0416/ag_news
- Zhang, X., Zhao, J., & LeCun, Y. (2015). *Character-level Convolutional Networks for Text Classification*. NIPS 2015.
- Apache Spark MLlib documentation: https://spark.apache.org/docs/latest/ml-guide.html
- Gradio documentation: https://gradio.app/guides/quickstart
